In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt


In [2]:
def create_board():
    
    return [[' ' for _ in range(3)] for _ in range(3)]


In [ ]:
def print_board(board):
    for r in range(3):
        print("  | ".join(board[r] ))
        if r < 2:
            print("---+----+---")
board = create_board()
print_board(board)


   |    |  
---+----+---
   |    |  
---+----+---
   |    |  


In [5]:
def available_moves(board): #lediga rutor
    moves = []
    for r in range(3):
        for c in range(3):
            if board[r][c] == " ":
                moves.append((r, c))
    return moves


In [6]:
def make_move(board, move, player): #gör ett drag
    r, c = move
    if board[r][c] != " ":
        return False
    board[r][c] = player
    return True


In [7]:
def check_winner(board): #kollar vinnare
    lines = []

    # rader
    for r in range(3):
        lines.append(board[r])

    # kolumner
    for c in range(3):
        lines.append([board[0][c], board[1][c], board[2][c]])

    # diagonaler
    lines.append([board[0][0], board[1][1], board[2][2]])
    lines.append([board[0][2], board[1][1], board[2][0]])

    # vinst
    for line in lines:
        if line[0] != " " and line[0] == line[1] == line[2]:
            return line[0]  # 'O' eller 'X'

    # draw
    if len(available_moves(board)) == 0:
        return "draw"

    return None


In [8]:
def random_player(board, player): #Random spelare
    return random.choice(available_moves(board))


In [9]:
def play_game(player_O_func, player_X_func, verbose=True): #Random vs Random matchloop
    board = create_board()
    current = "O"

    if verbose:
        print_board(board)
        print()

    while True:
        move = player_O_func(board, "O") if current == "O" else player_X_func(board, "X")

        ok = make_move(board, move, current)
        if not ok:
            # ogiltigt drag = förlust
            winner = "X" if current == "O" else "O"
            if verbose:
                print(f"{current} gjorde ogiltigt drag {move} -> {winner} vinner!")
            return winner

        if verbose:
            print(f"{current} spelar {move}")
            print_board(board)
            print()

        result = check_winner(board)
        if result is not None:
            return result

        current = "X" if current == "O" else "O"


In [10]:
play_game(random_player, random_player, verbose=True)


   |    |  
---+----+---
   |    |  
---+----+---
   |    |  

O spelar (0, 0)
O  |    |  
---+----+---
   |    |  
---+----+---
   |    |  

X spelar (2, 2)
O  |    |  
---+----+---
   |    |  
---+----+---
   |    | X

O spelar (0, 2)
O  |    | O
---+----+---
   |    |  
---+----+---
   |    | X

X spelar (2, 1)
O  |    | O
---+----+---
   |    |  
---+----+---
   | X  | X

O spelar (1, 0)
O  |    | O
---+----+---
O  |    |  
---+----+---
   | X  | X

X spelar (1, 2)
O  |    | O
---+----+---
O  |    | X
---+----+---
   | X  | X

O spelar (1, 1)
O  |    | O
---+----+---
O  | O  | X
---+----+---
   | X  | X

X spelar (2, 0)
O  |    | O
---+----+---
O  | O  | X
---+----+---
X  | X  | X



'X'

In [11]:
def human_player(board, player):
    while True:
        try:
            choice = int(input(f"Player {player}, choose position (1-9): "))

            if choice < 1 or choice > 9:
                print("Please choose a number between 1 and 9.")
                continue

            r = (choice - 1) // 3
            c = (choice - 1) % 3
            move = (r, c)

            if move in available_moves(board):
                return move
            else:
                print("That position is already taken. Try again.")

        except ValueError:
            print("Please enter a valid number (1-9).")


In [12]:
play_game(human_player, human_player, verbose=True)


   |    |  
---+----+---
   |    |  
---+----+---
   |    |  

O spelar (0, 0)
O  |    |  
---+----+---
   |    |  
---+----+---
   |    |  

X spelar (0, 1)
O  | X  |  
---+----+---
   |    |  
---+----+---
   |    |  

O spelar (1, 0)
O  | X  |  
---+----+---
O  |    |  
---+----+---
   |    |  

X spelar (1, 1)
O  | X  |  
---+----+---
O  | X  |  
---+----+---
   |    |  

O spelar (2, 0)
O  | X  |  
---+----+---
O  | X  |  
---+----+---
O  |    |  



'O'

In [13]:
def is_winning_move(board, move, player):
    r, c = move
    board[r][c] = player
    win = check_winner(board) == player
    board[r][c] = " "
    return win


In [14]:
def policy_player(board, player):
    opponent = "X" if player == "O" else "O"
    moves = available_moves(board)

    # 1. Win if possible
    for move in moves:
        if is_winning_move(board, move, player):
            return move

    # 2. Block opponent win
    for move in moves:
        if is_winning_move(board, move, opponent):
            return move

    # 3. Take center
    if (1, 1) in moves:
        return (1, 1)

    # 4. Take a corner
    corners = [(0,0), (0,2), (2,0), (2,2)]
    available_corners = [c for c in corners if c in moves]
    if available_corners:
        return random.choice(available_corners)

    # 5. Otherwise random
    return random.choice(moves)


In [15]:
play_game(human_player, policy_player, verbose=True)


   |    |  
---+----+---
   |    |  
---+----+---
   |    |  

O spelar (0, 0)
O  |    |  
---+----+---
   |    |  
---+----+---
   |    |  

X spelar (1, 1)
O  |    |  
---+----+---
   | X  |  
---+----+---
   |    |  

O spelar (0, 2)
O  |    | O
---+----+---
   | X  |  
---+----+---
   |    |  

X spelar (0, 1)
O  | X  | O
---+----+---
   | X  |  
---+----+---
   |    |  

O spelar (2, 1)
O  | X  | O
---+----+---
   | X  |  
---+----+---
   | O  |  

X spelar (2, 2)
O  | X  | O
---+----+---
   | X  |  
---+----+---
   | O  | X

O spelar (2, 0)
O  | X  | O
---+----+---
   | X  |  
---+----+---
O  | O  | X

X spelar (1, 0)
O  | X  | O
---+----+---
X  | X  |  
---+----+---
O  | O  | X

O spelar (1, 2)
O  | X  | O
---+----+---
X  | X  | O
---+----+---
O  | O  | X



'draw'